# 03 Feature Engineering

This notebook turns the cleaned Telco Customer Churn data into model-ready features.

In the MLOps lifecycle, feature engineering is the bridge between business data and model training. The goal is not just to transform columns technically, but to make relevant patterns usable for a model while avoiding data leakage.

- En este notebook transformamos el dataset limpio en un dataset preparado para entrenar un modelo de Machine Learning.
- Feature Engineering convierte los datos del negocio en una representación adecuada para que el modelo pueda aprender.
- El objetivo no es modificar columnas sin motivo, sino facilitar que el modelo encuentre patrones útiles.

## Feature Engineering Goals

This notebook demonstrates:

- Separating identifiers, target columns, numeric features, and categorical features. --> Primero clasificamos las columnas según su función para poder aplicar el preprocesamiento adecuado.
- Creating a reproducible train/test split before fitting transformations. --> Antes de transformar los datos se realiza la división entre entrenamiento y prueba para evitar Data Leakage
- Building a scikit-learn `ColumnTransformer`. --> Se utiliza ColumnTransformer para aplicar automáticamente distintas transformaciones según el tipo de columna.
- Imputing and scaling numeric features. --> Las variables numéricas se limpian y posteriormente se escalan para que tengan una magnitud similar.>
- Imputing and one-hot encoding categorical features. --> Las variables categóricas se completan cuando es necesario y se convierten en variables numéricas mediante One-Hot Encoding.
- Exporting processed train/test data for the next notebook. --> Al finalizar se guardan los datos procesados para utilizarlos en el siguiente notebook.

We still keep all logic in the notebook. Refactoring into `src/` comes later. --> De momento toda la lógica permanece dentro del notebook; más adelante se organizará en módulos dentro de src/.

## Leakage Reminder

Feature transformations can leak information if they are fitted on the full dataset before the train/test split. --> Las transformaciones deben aprender únicamente de los datos de entrenamiento para evitar que el modelo utilice información del conjunto de prueba.

For example:

- Imputation values should be learned only from the training data.
- Scaling parameters should be learned only from the training data.
- Encoded category structure should be created from the training data and then applied to test data.

Therefore, this notebook splits the data first and fits the preprocessing pipeline only on `X_train`.

Primero se divide el dataset y únicamente después se ajusta el pipeline de preprocesamiento utilizando el conjunto de entrenamiento.

## 1. Setup

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer #Permite aplicar transformaciones diferentes según el tipo de columna.
from sklearn.impute import SimpleImputer #Imputer = rellenador de datos faltantes.
from sklearn.model_selection import train_test_split #divide Train/Test.
from sklearn.pipeline import Pipeline #"Haz estos pasos siempre en este orden."
from sklearn.preprocessing import OneHotEncoder, StandardScaler #Hace que todas las columnas numéricas tengan una escala parecida.

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

RANDOM_STATE = 42
TEST_SIZE = 0.2


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "interim" / "telco_churn_cleaned.csv").exists():
            return path
    raise FileNotFoundError("Could not find project root with data/interim/telco_churn_cleaned.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "telco_churn_cleaned.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned data path: {INTERIM_DATA_PATH}")

Project root: /home/patri/master/2026-06-ads-II-master
Cleaned data path: /home/patri/master/2026-06-ads-II-master/data/interim/telco_churn_cleaned.csv


### Conceptos nuevos

- **train_test_split** → divide el dataset en Train y Test.
- **SimpleImputer** → rellena valores faltantes (NaN).
- **StandardScaler** → pone las variables numéricas en la misma escala.
- **OneHotEncoder** → convierte variables categóricas en columnas binarias (0/1).
- **Pipeline** → une todos los pasos del preprocesamiento para ejecutarlos siempre en el mismo orden.

## 2. Load Cleaned Data

In [3]:
cleaned_df = pd.read_csv(INTERIM_DATA_PATH)

print(f"Rows: {cleaned_df.shape[0]:,}")
print(f"Columns: {cleaned_df.shape[1]:,}")
cleaned_df.head()

Rows: 7,043
Columns: 22


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBinary
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


TotalCharges se convirtió de texto (object) a un valor numérico (float).  
Los valores vacíos de TotalCharges para clientes con tenure = 0 se sustituyeron por 0.0.  
Se creó una nueva variable objetivo ChurnBinary (No → 0, Yes → 1).  
Se validó que no existieran filas duplicadas ni customerID duplicados.  
Se comprobó que la variable Churn solo contuviera los valores esperados (Yes y No).  
El dataset limpio se guardó como:  
data/interim/telco_churn_cleaned.csv  

In [4]:
cleaned_df.dtypes.rename("dtype").to_frame()

,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


¿Qué tenemos ahora?  
 
El dataset ya está limpio, pero todavía no está preparado para entrenar un modelo.  

Todavía faltan varios pasos de Feature Engineering:  

- Separar variables predictoras (X) y variable objetivo (y).  
- Dividir los datos en Train y Test.  
- Imputar posibles valores faltantes.  
- Escalar las variables numéricas.  
- Codificar las variables categóricas mediante One-Hot Encoding.  
- Guardar el resultado final como dataset processed.  

## Imputation Reflection

Imputation is a common way to make an ML pipeline robust, but it is not automatically the best business decision.

For model training, one valid option can be to exclude rows with missing values if the number of affected observations is small and if this exclusion is defensible. In production, however, new observations can arrive with missing fields. If the inference pipeline has no strategy for this case, a single missing value can break the prediction endpoint.

In this notebook, imputation is included for demonstration purposes and to prepare a robust training/inference pattern. The concrete imputation strategy still needs a strong domain justification:

- What does a missing value mean in the business process?
- Is it truly unknown, not applicable, not collected, or a data quality problem?
- Should we impute, flag, reject the request, or route it to manual review?

The technical pipeline can keep the system running, but the decision must be explained and monitored.

##### Observacion

La imputación consiste en rellenar valores que faltan para que el modelo pueda seguir funcionando.

Sin embargo, imputar un dato no siempre es la mejor decisión. Antes hay que entender por qué falta ese valor.

Un valor faltante puede significar:

El dato es desconocido.
No aplica para ese cliente.
No se recogió correctamente.
Existe un problema de calidad de los datos.

En producción siempre pueden llegar nuevos registros con valores vacíos. Por eso el pipeline debe tener una estrategia para tratar esos casos y evitar que el modelo falle.

Idea principal: la imputación no es solo una decisión técnica, también es una decisión de negocio.

## 3. Create Business Features

Before encoding and scaling, we add a small set of interpretable business features.

These features translate domain hypotheses into columns:

- `TenureGroup`: customer lifecycle stage.
- `AverageMonthlyCharges`: historical average monthly charge.
- `MonthlyChargesDelta`: current monthly charge compared with the historical average.
- `NumberOfAddOnServices`: breadth of additional service usage.

Important inference-time rule: these calculations must be identical during training and prediction. Later, when we refactor into `src/`, this logic must move into reusable code that is called by both training and the API.


##### Observacion

En esta parte no solo se limpian los datos, sino que se crean nuevas variables con significado de negocio.

El objetivo es generar información que pueda ayudar al modelo a encontrar mejores patrones.

Las nuevas variables representan ideas como:

Antigüedad del cliente.
Gasto medio mensual.
Diferencia entre el gasto actual y el histórico.
Número de servicios adicionales contratados.

Estas variables suelen aportar más información que utilizar únicamente las columnas originales.

In [5]:
ADD_ON_SERVICE_COLUMNS = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]


def create_business_features(df: pd.DataFrame) -> pd.DataFrame:
    # These features are deterministic row-level transformations. They do not
    # learn parameters from the full dataset, so they can be applied before the split.
    featured = df.copy()

    featured["TenureGroup"] = pd.cut(
        featured["tenure"],
        bins=[-1, 6, 12, 24, 48, float("inf")],
        labels=["0-6 months", "7-12 months", "13-24 months", "25-48 months", "49+ months"],
    ).astype("object") ## Se agrupa la antigüedad del cliente en intervalos para representar mejor el ciclo de vida del cliente y facilitar que el modelo detecte patrones.

    featured["AverageMonthlyCharges"] = featured["TotalCharges"] / featured["tenure"]
    featured.loc[featured["tenure"] == 0, "AverageMonthlyCharges"] = featured.loc[
        featured["tenure"] == 0,
        "MonthlyCharges",
    ] ## Se calcula el gasto mensual medio del cliente. Si el cliente acaba de llegar (tenure = 0), se utiliza MonthlyCharges para evitar dividir entre cero.

    featured["MonthlyChargesDelta"] = (
        featured["MonthlyCharges"] - featured["AverageMonthlyCharges"]
    ) # Calcula cuánto difiere la cuota mensual actual respecto al promedio histórico del cliente.

    featured["NumberOfAddOnServices"] = (
        featured[ADD_ON_SERVICE_COLUMNS]
        .eq("Yes")
        .sum(axis=1)
    ) # Cuenta el número de servicios adicionales contratados por cada cliente. Un mayor número de servicios puede indicar un cliente más vinculado a la empresa.

    return featured


featured_df = create_business_features(cleaned_df)
featured_df[
    [
        "customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "TenureGroup",
        "AverageMonthlyCharges",
        "MonthlyChargesDelta",
        "NumberOfAddOnServices",
    ]
].head(10)

,customerID,tenure,MonthlyCharges,TotalCharges,TenureGroup,AverageMonthlyCharges,MonthlyChargesDelta,NumberOfAddOnServices
0,7590-VHVEG,1,29.85,29.85,0-6 months,29.850000,0.000000,1
1,5575-GNVDE,34,56.95,1889.50,25-48 months,55.573529,1.376471,2
2,3668-QPYBK,2,53.85,108.15,0-6 months,54.075000,-0.225000,2
3,7795-CFOCW,45,42.30,1840.75,25-48 months,40.905556,1.394444,3
4,9237-HQITU,2,70.70,151.65,0-6 months,75.825000,-5.125000,0
5,9305-CDSKC,8,99.65,820.50,7-12 months,102.562500,-2.912500,3
6,1452-KIOVK,22,89.10,1949.40,13-24 months,88.609091,0.490909,2
7,6713-OKOMC,10,29.75,301.90,7-12 months,30.190000,-0.440000,1
8,7892-POOKP,28,104.80,3046.05,25-48 months,108.787500,-3.987500,4
9,6388-TABGU,62,56.15,3487.95,49+ months,56.257258,-0.107258,2


Has visto cuatro técnicas clásicas de Feature Engineering:  

Crear categorías a partir de variables numéricas (pd.cut).  
Crear promedios (AverageMonthlyCharges).  
Crear diferencias (MonthlyChargesDelta).  
Crear contadores (NumberOfAddOnServices).  

### Feature Logic and Inference Time

At inference time, a new customer record must first go through the same feature creation logic:

1. Derive `AverageMonthlyCharges` from that customer's `TotalCharges` and `tenure`.
2. Derive `MonthlyChargesDelta` from that customer's current bill and historical average.
3. Count active add-on services for that customer.
4. Pass the resulting feature row into the fitted preprocessing pipeline.

The fitted pipeline stores learned preprocessing parameters, such as median imputation values, scaling means and standard deviations, and known one-hot categories. It does not store customer-specific engineered feature values.

This is why feature logic and the fitted preprocessor must be versioned together.

Lógica de las features en inferencia

Las mismas transformaciones que usamos para entrenar el modelo deben aplicarse exactamente igual cuando llegue un cliente nuevo.

No se pueden crear las variables de una forma durante el entrenamiento y de otra distinta en producción.

El pipeline de preprocesamiento guarda toda la información que ha aprendido durante el entrenamiento (medianas, medias, desviaciones estándar, categorías, etc.) y reutiliza esos mismos valores para transformar datos nuevos.

Por eso la lógica de creación de variables y el pipeline entrenado deben guardarse y versionarse juntos.

In [6]:
engineered_features = [
    "TenureGroup",
    "AverageMonthlyCharges",
    "MonthlyChargesDelta",
    "NumberOfAddOnServices",
]

featured_df[engineered_features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TenureGroup,7043,5,49+ months,2239,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AverageMonthlyCharges,7043.0,NaN,NaN,NaN,64.762906,30.189796,13.775,35.935156,70.3375,90.174158,121.4
MonthlyChargesDelta,7043.0,NaN,NaN,NaN,-0.001213,2.614121,-18.9,-1.159091,0.0,1.145567,19.125
NumberOfAddOnServices,7043.0,NaN,NaN,NaN,2.03791,1.847682,0.0,0.0,2.0,3.0,6.0


## Feature Selection Context

Feature selection is not the main focus of this module, but it is an important modeling decision.

In this notebook, we start pragmatically: we keep features that are technically valid, available at prediction time, and plausible from a business perspective. This is a reasonable first baseline strategy because it keeps the workflow understandable and avoids premature optimization.

Still, not every engineered feature should automatically be used forever. Feature selection can be useful when features are:

- redundant, for example when one feature is directly derived from another;
- unstable, for example when values may not be available reliably at inference time;
- hard to justify from a business or governance perspective;
- harmful for interpretability, especially in linear models;
- not useful according to later model evaluation.

For the first baseline, we keep the selection simple and transparent. Later, model evaluation and error analysis can guide whether some features should be removed, grouped differently, or replaced.

## 4. Define Feature Groups

In [7]:
ID_COLUMNS = ["customerID"]
TARGET_COLUMNS = ["Churn", "ChurnBinary"]

NUMERIC_FEATURES = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "AverageMonthlyCharges",
    "MonthlyChargesDelta",
    "NumberOfAddOnServices",
]

BINARY_FEATURES = [
    "SeniorCitizen",
]

CATEGORICAL_FEATURES = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TenureGroup",
]

FEATURE_COLUMNS = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

feature_overview = pd.DataFrame({
    "group": (
        ["numeric_continuous"] * len(NUMERIC_FEATURES)
        + ["binary_indicator"] * len(BINARY_FEATURES)
        + ["categorical"] * len(CATEGORICAL_FEATURES)
    ),
    "column": FEATURE_COLUMNS,
})

feature_overview

,group,column
0,numeric_continuous,tenure
1,numeric_continuous,MonthlyCharges
2,numeric_continuous,TotalCharges
3,numeric_continuous,AverageMonthlyCharges
4,numeric_continuous,MonthlyChargesDelta
5,numeric_continuous,NumberOfAddOnServices
6,binary_indicator,SeniorCitizen
7,categorical,gender
8,categorical,Partner
9,categorical,Dependents


### Feature Selection Note: `tenure` and `TenureGroup`

`TenureGroup` is derived directly from `tenure`, so both features contain related information.

Keeping both can still be useful in an early baseline:

- `tenure` preserves the exact number of months.
- `TenureGroup` gives the model coarse lifecycle segments, such as new customers or long-term customers.

However, this is a feature selection decision, not a rule. Keeping both can introduce redundancy. For some models this is acceptable, while for interpretable linear models it can make coefficients harder to explain.

In this notebook, we keep both for the first baseline because they express different views of the same business concept: exact customer age and customer lifecycle stage. Later model evaluation can compare whether both are needed.

In [8]:
missing_features = set(FEATURE_COLUMNS + TARGET_COLUMNS + ID_COLUMNS) - set(featured_df.columns)
if missing_features:
    raise ValueError(f"Missing expected columns: {sorted(missing_features)}")

print(f"Number of model features before encoding: {len(FEATURE_COLUMNS)}")
print(f"Identifier columns excluded from modeling: {ID_COLUMNS}")
print(f"Target columns excluded from features: {TARGET_COLUMNS}")

Number of model features before encoding: 23
Identifier columns excluded from modeling: ['customerID']
Target columns excluded from features: ['Churn', 'ChurnBinary']


Selección de variables (Feature Selection)  

En este primer modelo se mantienen todas las variables que parecen útiles.  
Algunas variables contienen información parecida (por ejemplo, tenure y TenureGroup).  
Más adelante se evaluará si merece la pena conservar ambas o eliminar alguna.  
Antes de entrenar el modelo se comprueba que todas las columnas esperadas existen (fail fast).  
customerID no se utiliza para entrenar porque solo identifica al cliente.  
Churn y ChurnBinary no son variables de entrada, sino la variable objetivo que el modelo debe aprender a predecir.  

## 5. Create X and y

In [9]:
X = featured_df[FEATURE_COLUMNS].copy()
y = featured_df["ChurnBinary"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
y.value_counts(normalize=True).mul(100).round(2).rename("target_share_percent")

X shape: (7043, 23)
y shape: (7043,)


ChurnBinary
0    73.46
1    26.54
Name: target_share_percent, dtype: float64

Creación de X e y  
  
Antes de entrenar un modelo de Machine Learning se separan los datos en:  

X: variables de entrada (features) que el modelo utilizará para hacer la predicción.  
y: variable objetivo (target) que el modelo debe aprender a predecir.  

En este proyecto, X contiene las 23 variables seleccionadas (numéricas, binarias y categóricas), mientras que y contiene únicamente ChurnBinary (0 = No, 1 = Sí).  

Las columnas customerID y Churn no forman parte de X:  
 
customerID solo identifica al cliente y no aporta información útil para predecir.  
Churn contiene la respuesta correcta y usarla como entrada produciría data leakage, ya que el modelo conocería de antemano el resultado que intenta predecir.  

## 6. Train/Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (5634, 23)
X_test:  (1409, 23)
y_train: (5634,)
y_test:  (1409,)


In [11]:
target_split_check = pd.DataFrame({
    "train_percent": y_train.value_counts(normalize=True).mul(100).round(2),
    "test_percent": y_test.value_counts(normalize=True).mul(100).round(2),
})

target_split_check

,train_percent,test_percent
ChurnBinary,,
0,73.46,73.46
1,26.54,26.54


Train/Test Split  

Antes de entrenar un modelo se divide el dataset en dos partes:  

Train: datos que el modelo utilizará para aprender.  
Test: datos nuevos que el modelo nunca verá durante el entrenamiento y que servirán para evaluar si realmente ha aprendido a generalizar.  
  
En este proyecto se utiliza un 80 % para entrenamiento y un 20 % para test (test_size = 0.2).  

random_state = 42 fija la semilla aleatoria para que el reparto sea siempre el mismo y el experimento sea reproducible.  

stratify = y mantiene la misma proporción de clases (No/Sí) tanto en Train como en Test, evitando que uno de los conjuntos quede desbalanceado.  

## 7. Build Preprocessing Pipeline

In [ ]:
def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("binary", binary_pipeline, BINARY_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )


preprocessor = build_preprocessor()
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

Build Preprocessing Pipeline  

En esta etapa se construye un pipeline de preprocesamiento que preparará automáticamente los datos antes de entrenar el modelo.  

Cada tipo de variable necesita un tratamiento diferente:  

- Variables numéricas: rellenar valores faltantes con la mediana y aplicar StandardScaler.  
- Variables binarias: rellenar valores faltantes con el valor más frecuente.  
- Variables categóricas: rellenar valores faltantes y convertir el texto en columnas numéricas mediante One-Hot Encoding.  

Pipeline permite ejecutar varios pasos siempre en el mismo orden.  

ColumnTransformer aplica automáticamente el pipeline adecuado a cada grupo de columnas.  

En esta celda todavía no se aprende nada (fit). Solo se define la receta de transformación que se utilizará más adelante.  

Pipeline de preprocesamiento  

- SimpleImputer(strategy="median"): si una variable numérica tiene un valor faltante (NaN), se sustituye por la mediana de esa columna calculada usando únicamente los datos de entrenamiento. La mediana suele ser más robusta que  la media frente a valores extremos.  
- StandardScaler(): cambia la escala de las variables numéricas para que todas tengan magnitudes similares. No cambia el significado de los datos, solo su escala, lo que ayuda a muchos modelos a entrenar mejor.  
SimpleImputer(strategy="most_frequent"): para variables binarias o categóricas, los valores faltantes se sustituyen por el valor que aparece con mayor frecuencia. Es una decisión práctica, aunque en un problema real debe   justificarse desde el punto de vista del negocio.  
- OneHotEncoder(): transforma variables de texto (por ejemplo, Contract) en varias columnas numéricas de 0 y 1, ya que los modelos de Machine Learning no pueden trabajar directamente con texto.  
- handle_unknown="ignore": si durante la predicción aparece una categoría nueva que no existía en el entrenamiento, el pipeline no genera un error y continúa funcionando.  
- sparse_output=False: devuelve el resultado como una matriz normal, más fácil de inspeccionar en el notebook.  

## 8. Fit on Training Data and Transform Both Splits

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"Processed X_test shape:  {X_test_processed.shape}")

Processed X_train shape: (5634, 53)
Processed X_test shape:  (1409, 53)


In [14]:
X_train_processed

array([[ 0.10237124, -0.52197565, -0.2622572 , ...,  1.        ,
         0.        ,  0.        ],
       [-0.71174346,  0.33747781, -0.50363479, ...,  0.        ,
         0.        ,  0.        ],
       [-0.79315493, -0.80901319, -0.74988292, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.30468611,  1.25666162,  0.15834357, ...,  1.        ,
         0.        ,  0.        ],
       [-0.34539184, -1.47766135, -0.79707463, ...,  0.        ,
         0.        ,  0.        ],
       [-1.07809507, -1.46936546, -0.96096216, ...,  0.        ,
         0.        ,  0.        ]], shape=(5634, 53))

In [15]:
feature_names = preprocessor.get_feature_names_out()

processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)
processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index,
)

processed_train_df.head()

,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,numeric__AverageMonthlyCharges,numeric__MonthlyChargesDelta,numeric__NumberOfAddOnServices,binary__SeniorCitizen,categorical__gender_Female,categorical__gender_Male,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,categorical__Dependents_Yes,categorical__PhoneService_No,categorical__PhoneService_Yes,categorical__MultipleLines_No,categorical__MultipleLines_No phone service,categorical__MultipleLines_Yes,categorical__InternetService_DSL,categorical__InternetService_Fiber optic,categorical__InternetService_No,categorical__OnlineSecurity_No,categorical__OnlineSecurity_No internet service,categorical__OnlineSecurity_Yes,categorical__OnlineBackup_No,categorical__OnlineBackup_No internet service,categorical__OnlineBackup_Yes,categorical__DeviceProtection_No,categorical__DeviceProtection_No internet service,categorical__DeviceProtection_Yes,categorical__TechSupport_No,categorical__TechSupport_No internet service,categorical__TechSupport_Yes,categorical__StreamingTV_No,categorical__StreamingTV_No internet service,categorical__StreamingTV_Yes,categorical__StreamingMovies_No,categorical__StreamingMovies_No internet service,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check,categorical__TenureGroup_0-6 months,categorical__TenureGroup_13-24 months,categorical__TenureGroup_25-48 months,categorical__TenureGroup_49+ months,categorical__TenureGroup_7-12 months
3738,0.102371,-0.521976,-0.262257,-0.539986,0.226917,0.507935,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3151,-0.711743,0.337478,-0.503635,0.391496,-0.639564,-0.570530,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4860,-0.793155,-0.809013,-0.749883,-0.646102,-1.867855,0.507935,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3867,-0.263980,0.284384,-0.172722,0.276552,0.081601,1.047168,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3810,-1.281624,-0.676279,-0.989374,-0.674608,0.003149,-1.109762,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


Fit and Transform  

En esta celda se ajusta el preprocesador únicamente con X_train y después se transforma tanto X_train como X_test.  

fit_transform(X_train) aprende los parámetros necesarios del conjunto de entrenamiento y aplica las transformaciones.  

transform(X_test) aplica las mismas transformaciones al conjunto de test sin volver a aprender nada nuevo.  

Esto evita Data Leakage, porque el conjunto de test no participa en el aprendizaje del preprocesamiento.  

El número de columnas aumenta de 23 a 53 porque las variables categóricas se convierten en varias columnas mediante One-Hot Encoding.  

In [16]:
pd.DataFrame({
    "feature_name": feature_names,
}).head(40)

,feature_name
0,numeric__tenure
1,numeric__MonthlyCharges
2,numeric__TotalCharges
3,numeric__AverageMonthlyCharges
4,numeric__MonthlyChargesDelta
5,numeric__NumberOfAddOnServices
6,binary__SeniorCitizen
7,categorical__gender_Female
8,categorical__gender_Male
9,categorical__Partner_No


## 9. Validate Processed Features

In [17]:
validation = {
    "train_missing_values": int(processed_train_df.isna().sum().sum()),
    "test_missing_values": int(processed_test_df.isna().sum().sum()),
    "train_rows": processed_train_df.shape[0],
    "test_rows": processed_test_df.shape[0],
    "processed_features": processed_train_df.shape[1],
}

validation

{'train_missing_values': 0,
 'test_missing_values': 0,
 'train_rows': 5634,
 'test_rows': 1409,
 'processed_features': 53}

In [18]:
processed_train_df.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
numeric__tenure,5634.0,-1.008935e-17,1.000089,-1.322329,-0.955978,-0.141863,0.916486,1.608483
numeric__MonthlyCharges,5634.0,-2.402527e-16,1.000089,-1.544028,-0.971198,0.184834,0.831912,1.785939
numeric__TotalCharges,5634.0,2.522338e-17,1.000089,-1.008922,-0.832101,-0.396845,0.674194,2.801869
numeric__AverageMonthlyCharges,5634.0,-2.219658e-16,1.000089,-1.691127,-0.956007,0.186966,0.844621,1.868225
numeric__MonthlyChargesDelta,5634.0,-1.008935e-17,1.000089,-7.270674,-0.449612,0.003149,0.439322,7.363566
numeric__NumberOfAddOnServices,5634.0,0.000000e+00,1.000089,-1.109762,-1.109762,-0.031297,0.507935,2.125633
binary__SeniorCitizen,5634.0,1.632943e-01,0.369667,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__gender_Female,5634.0,4.971601e-01,0.500036,0.000000,0.000000,0.000000,1.000000,1.000000
categorical__gender_Male,5634.0,5.028399e-01,0.500036,0.000000,0.000000,1.000000,1.000000,1.000000
categorical__Partner_No,5634.0,5.156195e-01,0.499800,0.000000,0.000000,1.000000,1.000000,1.000000


Validación de las variables procesadas  

Se comprueba que el preprocesamiento ha funcionado correctamente antes de continuar.  

Se verifica que:  

- No quedan valores faltantes.  
- El número de filas es correcto.  
- Las nuevas variables se han creado correctamente. 
- Las variables numéricas han sido escaladas.  
- Las variables categóricas ya están codificadas en formato 0/1.  

## 10. Save Processed Data for the Baseline Notebook

In [19]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_train_df.to_csv(PROCESSED_DIR / "X_train_processed.csv", index=False)
processed_test_df.to_csv(PROCESSED_DIR / "X_test_processed.csv", index=False)
y_train.to_frame("ChurnBinary").to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_frame("ChurnBinary").to_csv(PROCESSED_DIR / "y_test.csv", index=False)

pd.Series(feature_names, name="feature_name").to_csv(
    PROCESSED_DIR / "feature_names.csv",
    index=False,
)

metadata_columns = [
    "customerID",
    "Contract",
    "TenureGroup",
    "InternetService",
    "PaymentMethod",
    "SeniorCitizen",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
    "ChurnBinary",
]

train_metadata = featured_df.loc[X_train.index, metadata_columns].copy()
test_metadata = featured_df.loc[X_test.index, metadata_columns].copy()
train_metadata.insert(0, "row_id", range(len(train_metadata)))
test_metadata.insert(0, "row_id", range(len(test_metadata)))

train_metadata.to_csv(PROCESSED_DIR / "train_metadata.csv", index=False)
test_metadata.to_csv(PROCESSED_DIR / "test_metadata.csv", index=False)

print(f"Saved processed data to: {PROCESSED_DIR}")
print("Saved train/test metadata for traceability and error slicing.")

Saved processed data to: /home/patri/master/2026-06-ads-II-master/data/processed
Saved train/test metadata for traceability and error slicing.


##### Guardar los datos procesados  

Una vez terminado el preprocesamiento, se guardan los conjuntos Train y Test ya preparados para Machine Learning.  

Así el siguiente notebook puede empezar directamente a entrenar modelos sin repetir todo el proceso de limpieza y transformación.  

También se guardan metadatos para mantener la trazabilidad entre las filas originales y las procesadas.  

## Reflection: Pipeline Robustness and Input Contracts

The preprocessing pipeline is more robust than raw notebook code, but it is not a full production validation layer.

What the current pipeline handles well:

- Missing values inside existing numeric columns, for example a missing `MonthlyCharges` value, because the numeric pipeline uses median imputation.
- Missing values inside existing categorical columns, for example a missing `Contract` value, because the categorical pipeline uses most-frequent imputation.
- New categories in existing categorical columns, for example a new payment method value, because `OneHotEncoder(handle_unknown="ignore")` does not fail on unseen categories.

What the current pipeline does not handle:

- Missing input columns. If `TotalCharges` is not present at inference time, `AverageMonthlyCharges` and `MonthlyChargesDelta` cannot be calculated.
- Renamed columns. A request with `total_charges` instead of `TotalCharges` would fail unless we explicitly map or validate the schema.
- Invalid data types. A value such as `"twelve"` in `tenure` cannot be used safely without conversion and validation.
- Implausible values. Negative `MonthlyCharges`, negative `tenure`, or an unknown target label would indicate a data quality problem.
- Incomplete API requests. If a prediction request omits service columns such as `OnlineSecurity` or `TechSupport`, we cannot calculate `NumberOfAddOnServices` without a clear fallback rule.

This means that preprocessing and validation are related, but not the same thing.

The fitted preprocessing pipeline stores learned transformation parameters such as imputation values, scaling parameters, and one-hot categories. It does not define the full input contract for production. A production-ready application needs an explicit validation step before feature engineering and prediction.

Later, when we refactor this notebook into Python modules and build the API, we should define:

- Required input columns.
- Accepted data types.
- Allowed categorical values where appropriate.
- Plausible numeric ranges.
- Clear error messages for invalid requests.
- Business-approved fallback rules for missing or unknown values.

For the current notebook phase, the assumption is: all required raw input columns are available and named as expected. This is acceptable for learning the feature engineering workflow, but it is not enough for a robust inference service.

##### Robustez del pipeline

El pipeline ya puede manejar muchos casos comunes (valores faltantes, categorías nuevas, escalado...), pero todavía no valida completamente los datos de entrada.  

En un sistema real habría que comprobar antes que:  

- existen todas las columnas obligatorias;  
- los nombres son correctos;  
- los tipos de datos son válidos;  
- los valores tienen sentido desde el punto de vista del negocio.  

## Feature Engineering Summary

What this notebook prepared:

- `customerID` was excluded from model features.
- `Churn` and `ChurnBinary` were excluded from `X`.
- The target is `ChurnBinary`.
- Business features were added: `TenureGroup`, `AverageMonthlyCharges`, `MonthlyChargesDelta`, `NumberOfAddOnServices`.
- Continuous numeric features are imputed and scaled.
- Binary indicator features are imputed and kept as `0/1`.
- Categorical features are imputed and one-hot encoded.
- The preprocessor is fitted only on training data to avoid leakage.
- Processed train/test data was saved under `data/processed/`.
- Train/test metadata was saved for traceability and later error slicing.

Next notebook step: baseline model training.